# Core PCE Hybrid NKPC Decomposition: EXPINF1YR Full-Level Version, v2

This notebook builds a monthly retrospective decomposition of 12-month core PCE inflation into:

- a model-implied baseline / intercept
- inflation expectations from FRED `EXPINF1YR`
- demand pressure from the local `DEMAND` series in `Demand_Supply_Data.xlsx`
- supply pressure from the local `SUPPLY` series in `Demand_Supply_Data.xlsx`
- lagged inflation / persistence
- residual / other omitted factors

The decomposition is a regression-implied accounting identity, not a structural causal decomposition. This v2 notebook keeps the decomposition in inflation-rate levels: for each month, baseline plus expectations, demand, supply, lagged inflation, and residual equals the actual 12-month core PCE inflation rate.

## Workflow

1. Read the local FRED API key from `API Keys.txt`.
2. Download monthly FRED series: `PCEPILFE` and `EXPINF1YR`.
3. Read local monthly demand and supply series from `Demand_Supply_Data.xlsx`.
4. Build the dependent variable, demand proxy, supply proxy, lag-window averages, and lagged inflation.
5. Estimate the baseline lag-window hybrid NKPC using OLS with Newey-West/HAC standard errors.
6. Re-estimate the same regression for the full sample, start-to-January-2020 sample, and January-2020-to-latest sample, then compare coefficients.
7. Perform the Section 5-style variance decomposition based on each source's covariance contribution and the regression R-squared.
8. Plot a two-panel figure: actual/fitted inflation, then a full level decomposition whose components sum to actual inflation.
9. Decompose the 12-month change in core PCE inflation using 12-month changes in expectations, demand, supply, and lagged inflation.
10. Run diagnostics and robustness checks: separate demand/supply lag windows, distributed lags, correlated demand/supply proxies, residual persistence, and optional statsmodels verification.

In [ ]:
from pathlib import Path

# File locations
ROOT = Path.cwd()
API_KEYS_PATH = ROOT / "API Keys.txt"
DEMAND_SUPPLY_PATH = ROOT / "Demand_Supply_Data.xlsx"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# FRED series. This v2 notebook uses FRED for core PCE and expected inflation
# only; demand and supply are read from Demand_Supply_Data.xlsx.
FRED_SERIES = {
    "core_pce": "PCEPILFE",
    "exp_1yr": "EXPINF1YR",
}
EXPECTATIONS_SOURCE = "Cleveland Fed one-year-ahead expected inflation rate (FRED EXPINF1YR)"
DEMAND_SUPPLY_SOURCE = "Demand_Supply_Data.xlsx: DATE, DEMAND, SUPPLY"

# Main model settings
FRED_START = "1990-01-01"
ESTIMATION_MODE = "full_sample"  # retrospective full-sample coefficients
BASELINE_MODE = "sample_mean"  # "sample_mean" or "pre_pandemic_average"
PRE_PANDEMIC_START = "2000-01"
PRE_PANDEMIC_END = "2019-12"

# Coefficient comparison windows. January 2020 is included in both subsamples
# because the requested windows are start-to-January-2020 and January-2020-to-latest.
COEFFICIENT_SAMPLE_WINDOWS = [
    {"sample": "full_sample", "start": None, "end": None},
    {"sample": "start_to_2020_01", "start": None, "end": "2020-01"},
    {"sample": "2020_01_to_latest", "start": "2020-01", "end": None},
]

# Separate lag windows: demand is delayed, supply can pass through sooner.
DEMAND_MIN_LAG = 3
DEMAND_MAX_LAG = 9
SUPPLY_MIN_LAG = 0
SUPPLY_MAX_LAG = 6
HAC_LAGS = 12

# Policy/neutral benchmark choices used only when BASELINE_MODE == "pre_pandemic_average".
# Defaults remain data-based; set overrides explicitly if you want a policy benchmark.
EXPECTATIONS_STAR_OVERRIDE = None
PI_STAR_OVERRIDE = None

# Robustness settings
ROBUST_LAG_WINDOWS = [
    {"name": "demand_0_6_supply_0_6", "demand": (0, 6), "supply": (0, 6)},
    {"name": "demand_3_9_supply_0_6", "demand": (3, 9), "supply": (0, 6)},
    {"name": "demand_3_12_supply_0_9", "demand": (3, 12), "supply": (0, 9)},
    {"name": "demand_0_12_supply_0_12", "demand": (0, 12), "supply": (0, 12)},
]
DISTRIBUTED_LAG_SETS = {
    "simple_0_3_6": [0, 3, 6],
    "rich_0_to_6": list(range(0, 7)),
    "long_0_to_12": list(range(0, 13)),
}

In [ ]:
import json
import math
import textwrap
import urllib.parse
import urllib.request
from datetime import datetime

import numpy as np
import pandas as pd

try:
    get_ipython
except NameError:
    import matplotlib
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

try:
    from IPython.display import display
except ImportError:
    display = print

plt.rcParams.update({
    "figure.figsize": (13, 7),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

## Data Loading Helpers

In [ ]:
def load_api_keys(path: Path) -> dict:
    keys = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or ":" not in line:
            continue
        name, value = line.split(":", 1)
        keys[name.strip().upper()] = value.strip()
    return keys


def fetch_fred_series(series_id: str, api_key: str, observation_start: str = FRED_START) -> pd.Series:
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "observation_start": observation_start,
    }
    url = "https://api.stlouisfed.org/fred/series/observations?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url, timeout=45) as response:
        payload = json.loads(response.read().decode("utf-8"))

    if "observations" not in payload:
        raise RuntimeError(f"FRED response for {series_id} did not include observations: {payload}")

    obs = pd.DataFrame(payload["observations"])
    obs["date"] = pd.to_datetime(obs["date"])
    obs["value"] = pd.to_numeric(obs["value"].replace(".", np.nan), errors="coerce")
    series = obs.set_index(obs["date"].dt.to_period("M"))["value"].sort_index()
    series.name = series_id
    return series


def load_demand_supply_data(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path)
    normalized = {str(col).strip().lower(): col for col in raw.columns}

    date_col = next((col for name, col in normalized.items() if "date" in name), None)
    demand_col = next((col for name, col in normalized.items() if "demand" in name), None)
    supply_col = next((col for name, col in normalized.items() if "supply" in name), None)

    missing = [
        label
        for label, col in {"DATE": date_col, "DEMAND": demand_col, "SUPPLY": supply_col}.items()
        if col is None
    ]
    if missing:
        raise ValueError(f"Demand/supply workbook is missing required column(s): {missing}")

    out = raw[[date_col, demand_col, supply_col]].copy()
    out.columns = ["date", "demand", "supply"]
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["demand"] = pd.to_numeric(out["demand"], errors="coerce")
    out["supply"] = pd.to_numeric(out["supply"], errors="coerce")
    out = out.dropna(subset=["date"]).sort_values("date")
    out["month"] = out["date"].dt.to_period("M")
    out = out.groupby("month", as_index=True)[["demand", "supply"]].mean().sort_index()
    return out

## Download and Assemble Monthly Data

In [ ]:
if ESTIMATION_MODE != "full_sample":
    raise NotImplementedError(
        "This notebook currently implements retrospective full-sample estimation. "
        "Recursive or rolling real-time estimation should be added as a separate mode."
    )

api_keys = load_api_keys(API_KEYS_PATH)
fred_api_key = api_keys.get("FRED")
if not fred_api_key:
    raise ValueError("FRED API key was not found in API Keys.txt.")

fred_data = {
    name: fetch_fred_series(series_id, fred_api_key, observation_start=FRED_START)
    for name, series_id in FRED_SERIES.items()
}
demand_supply = load_demand_supply_data(DEMAND_SUPPLY_PATH)

raw = pd.concat(
    [
        fred_data["core_pce"].rename("core_pce_index"),
        fred_data["exp_1yr"].rename("exp_1yr"),
        demand_supply["demand"].rename("demand"),
        demand_supply["supply"].rename("supply"),
    ],
    axis=1,
).sort_index()

raw["pi_12m"] = 100 * (np.log(raw["core_pce_index"]) - np.log(raw["core_pce_index"].shift(12)))
raw["lag_pi_12m"] = raw["pi_12m"].shift(12)

print(f"Raw monthly panel: {raw.index.min()} to {raw.index.max()}")
print(f"Expectations source: {EXPECTATIONS_SOURCE}")
print(f"Demand/supply source: {DEMAND_SUPPLY_SOURCE}")
print("Main decomposition coefficients are estimated over the full available sample after dropping rows with missing model variables.")
print("A separate coefficient table below also estimates start-to-January-2020 and January-2020-to-latest subsamples.")
display(raw.tail(8))

## Model Utilities

The model uses centered regressors so each contribution is measured relative to an explicit benchmark. In this notebook, the default `sample_mean` baseline centers all regressors on estimation-sample averages. That keeps the full decomposition data-based and avoids making the main chart relative to a 2% target.

The optional `pre_pandemic_average` baseline centers expectations, demand, supply, and lagged inflation on the selected pre-pandemic window unless an override is explicitly set for expectations or lagged inflation.

Lag windows use separate minimum and maximum lags, so the baseline demand window can be delayed relative to the supply window.

In [ ]:
def lag_window_average(series: pd.Series, min_lag: int, max_lag: int) -> pd.Series:
    if min_lag < 0 or max_lag < min_lag:
        raise ValueError("Require 0 <= min_lag <= max_lag.")
    lagged = pd.concat([series.shift(lag) for lag in range(min_lag, max_lag + 1)], axis=1)
    return lagged.mean(axis=1, skipna=False)


def normal_pvalue_from_t(t_stat: float) -> float:
    if pd.isna(t_stat):
        return np.nan
    return math.erfc(abs(float(t_stat)) / math.sqrt(2.0))


def fit_ols_hac(frame: pd.DataFrame, y_col: str, x_cols: list, hac_lags: int = 12) -> dict:
    model_frame = frame[[y_col] + x_cols].dropna().copy()
    y = model_frame[y_col].astype(float).to_numpy()
    X_no_const = model_frame[x_cols].astype(float)
    X = pd.concat(
        [pd.Series(1.0, index=model_frame.index, name="const"), X_no_const],
        axis=1,
    )
    x_with_const = list(X.columns)
    X_values = X.to_numpy()

    beta = np.linalg.lstsq(X_values, y, rcond=None)[0]
    fitted_values = X_values @ beta
    residual_values = y - fitted_values

    nobs, nparams = X_values.shape
    xtx_inv = np.linalg.pinv(X_values.T @ X_values)

    xu = X_values * residual_values[:, None]
    s_matrix = xu.T @ xu
    max_lag = min(int(hac_lags), nobs - 1)
    for lag in range(1, max_lag + 1):
        weight = 1.0 - lag / (max_lag + 1.0)
        gamma = xu[lag:].T @ xu[:-lag]
        s_matrix += weight * (gamma + gamma.T)

    small_sample = nobs / max(nobs - nparams, 1)
    cov_hac = small_sample * xtx_inv @ s_matrix @ xtx_inv
    std_errors = np.sqrt(np.maximum(np.diag(cov_hac), 0.0))

    total_ss = float(((y - y.mean()) ** 2).sum())
    resid_ss = float((residual_values ** 2).sum())
    r2 = 1.0 - resid_ss / total_ss if total_ss > 0 else np.nan
    adj_r2 = 1.0 - (1.0 - r2) * (nobs - 1) / max(nobs - nparams, 1) if pd.notna(r2) else np.nan
    rmse = math.sqrt(resid_ss / max(nobs - nparams, 1))

    index = model_frame.index
    params = pd.Series(beta, index=x_with_const)
    se = pd.Series(std_errors, index=x_with_const)
    t_stats = params / se.replace(0, np.nan)
    p_values = t_stats.apply(normal_pvalue_from_t)
    summary = pd.DataFrame({
        "coef": params,
        "newey_west_se": se,
        "t_stat": t_stats,
        "p_value_normal_approx": p_values,
    })

    return {
        "params": params,
        "std_errors": se,
        "t_stats": t_stats,
        "p_values": p_values,
        "summary": summary,
        "fitted": pd.Series(fitted_values, index=index, name="fitted"),
        "residuals": pd.Series(residual_values, index=index, name="residual"),
        "r2": r2,
        "adj_r2": adj_r2,
        "rmse": rmse,
        "nobs": nobs,
        "hac_lags": hac_lags,
        "y_name": y_col,
        "x_names": x_cols,
    }


def pre_pandemic_mask(index: pd.PeriodIndex, start: str, end: str) -> np.ndarray:
    start_period = pd.Period(start, freq="M")
    end_period = pd.Period(end, freq="M")
    return (index >= start_period) & (index <= end_period)


def choose_baselines(
    frame: pd.DataFrame,
    sample_index: pd.PeriodIndex,
    demand_col: str,
    supply_col: str,
    baseline_mode: str = "sample_mean",
    expectations_col: str = "exp_1yr",
    pi_col: str = "pi_12m",
    pre_start: str = PRE_PANDEMIC_START,
    pre_end: str = PRE_PANDEMIC_END,
    expectations_override=EXPECTATIONS_STAR_OVERRIDE,
    pi_override=PI_STAR_OVERRIDE,
    extra_cols: list | None = None,
) -> dict:
    extra_cols = extra_cols or []
    baseline_mode = baseline_mode.lower().replace("+", "_")
    sample = frame.loc[sample_index]

    if baseline_mode == "sample_mean":
        baselines = {
            "E_star": sample[expectations_col].mean(),
            "D_star": sample[demand_col].mean(),
            "S_star": sample[supply_col].mean(),
            "PI_star": sample[pi_col].mean(),
            "mode": "sample_mean",
        }
        for col in extra_cols:
            baselines[f"{col}_star"] = sample[col].mean()
        return baselines

    if baseline_mode != "pre_pandemic_average":
        raise ValueError("BASELINE_MODE must be 'sample_mean' or 'pre_pandemic_average'.")

    window = frame.loc[pre_pandemic_mask(frame.index, pre_start, pre_end)]
    if window.empty:
        raise ValueError(f"No observations found in pre-pandemic benchmark window {pre_start} to {pre_end}.")

    baselines = {
        "E_star": window[expectations_col].mean() if expectations_override is None else float(expectations_override),
        "D_star": window[demand_col].mean(),
        "S_star": window[supply_col].mean(),
        "PI_star": window[pi_col].mean() if pi_override is None else float(pi_override),
        "mode": "pre_pandemic_average",
        "window": f"{pre_start} to {pre_end}",
    }
    for col in extra_cols:
        baselines[f"{col}_star"] = window[col].mean()
    return baselines


def baseline_interpretation_table(baselines: dict) -> pd.DataFrame:
    interpretation = {
        "E_star": "expectations benchmark",
        "D_star": "demand benchmark",
        "S_star": "supply benchmark",
        "PI_star": "lagged inflation benchmark",
        "mode": "baseline mode",
        "window": "benchmark window",
    }
    rows = []
    for key, value in baselines.items():
        rows.append({
            "item": key,
            "benchmark": value,
            "interpretation": interpretation.get(key, "additional benchmark"),
        })
    return pd.DataFrame(rows)


def format_benchmark_label(baselines: dict) -> str:
    if baselines.get("mode") == "sample_mean":
        return "E*, D*, S*, lagged pi*: estimation-sample means"
    return "E*, D*, S*, lagged pi*: pre-pandemic averages"


def filter_period_index(index: pd.PeriodIndex, start: str | None = None, end: str | None = None) -> pd.PeriodIndex:
    mask = pd.Series(True, index=index)
    if start is not None:
        mask &= index >= pd.Period(start, freq="M")
    if end is not None:
        mask &= index <= pd.Period(end, freq="M")
    return index[mask.to_numpy()]

## Baseline Lag-Window NKPC

Baseline specification:

`pi_t = alpha + beta_E * (E_t - E*) + beta_D * (Dbar_t - D*) + beta_S * (Sbar_t - S*) + rho * (pi_{t-12} - PI*) + eps_t`

where `Dbar_t` and `Sbar_t` are averages over explicit lag windows. The default demand window is 3-9 months, while the default supply window is 0-6 months.

The main decomposition continues to use the full-sample regression. The coefficient comparison table re-estimates this same specification over the configured subsample windows.

In [ ]:
def build_standard_contributions(result: dict, model_frame: pd.DataFrame) -> pd.DataFrame:
    contributions = pd.DataFrame(index=model_frame.index)
    contributions["baseline"] = result["params"]["const"]
    contributions["expectations"] = result["params"]["E_c"] * model_frame["E_c"]
    contributions["demand"] = result["params"]["D_c"] * model_frame["D_c"]
    contributions["supply"] = result["params"]["S_c"] * model_frame["S_c"]
    contributions["lagged_inflation"] = result["params"]["lag_pi_c"] * model_frame["lag_pi_c"]
    contributions["fitted"] = result["fitted"]
    contributions["actual"] = model_frame["pi_12m"]
    contributions["residual"] = contributions["actual"] - contributions["fitted"]
    contributions["identity_check"] = (
        contributions[["baseline", "expectations", "demand", "supply", "lagged_inflation", "residual"]].sum(axis=1)
        - contributions["actual"]
    )
    return contributions


def estimate_lag_window_model(
    data: pd.DataFrame,
    demand_min_lag: int = DEMAND_MIN_LAG,
    demand_max_lag: int = DEMAND_MAX_LAG,
    supply_min_lag: int = SUPPLY_MIN_LAG,
    supply_max_lag: int = SUPPLY_MAX_LAG,
    baseline_mode: str = BASELINE_MODE,
    hac_lags: int = HAC_LAGS,
    sample_start: str | None = None,
    sample_end: str | None = None,
    sample_label: str = "full_sample",
) -> dict:
    frame = data.copy()
    frame["Dbar"] = lag_window_average(frame["demand"], demand_min_lag, demand_max_lag)
    frame["Sbar"] = lag_window_average(frame["supply"], supply_min_lag, supply_max_lag)

    required = ["pi_12m", "exp_1yr", "Dbar", "Sbar", "lag_pi_12m"]
    preliminary_sample = frame[required].dropna()
    sample_index = filter_period_index(preliminary_sample.index, sample_start, sample_end)
    if len(sample_index) == 0:
        raise ValueError(f"No usable observations for sample {sample_label}: {sample_start} to {sample_end}.")

    baselines = choose_baselines(
        frame=frame,
        sample_index=sample_index,
        demand_col="Dbar",
        supply_col="Sbar",
        baseline_mode=baseline_mode,
    )

    frame["E_c"] = frame["exp_1yr"] - baselines["E_star"]
    frame["D_c"] = frame["Dbar"] - baselines["D_star"]
    frame["S_c"] = frame["Sbar"] - baselines["S_star"]
    frame["lag_pi_c"] = frame["lag_pi_12m"] - baselines["PI_star"]

    model_cols = ["E_c", "D_c", "S_c", "lag_pi_c"]
    model_frame = frame.loc[sample_index, ["pi_12m"] + model_cols].dropna()
    result = fit_ols_hac(model_frame, "pi_12m", model_cols, hac_lags=hac_lags)
    contributions = build_standard_contributions(result, model_frame)

    return {
        "frame": frame,
        "model_frame": model_frame,
        "result": result,
        "contributions": contributions,
        "baselines": baselines,
        "settings": {
            "estimation_mode": ESTIMATION_MODE,
            "sample_label": sample_label,
            "sample_start": sample_start,
            "sample_end": sample_end,
            "expectations_source": EXPECTATIONS_SOURCE,
            "demand_supply_source": DEMAND_SUPPLY_SOURCE,
            "demand_min_lag": demand_min_lag,
            "demand_max_lag": demand_max_lag,
            "supply_min_lag": supply_min_lag,
            "supply_max_lag": supply_max_lag,
            "baseline_mode": baseline_mode,
            "hac_lags": hac_lags,
        },
    }


baseline_model = estimate_lag_window_model(raw)
baseline_result = baseline_model["result"]
baseline_contributions = baseline_model["contributions"]
baseline_benchmarks = baseline_interpretation_table(baseline_model["baselines"])

print(f"Estimation sample: {baseline_contributions.index.min()} to {baseline_contributions.index.max()}")
print(f"Observations: {baseline_result['nobs']}, R-squared: {baseline_result['r2']:.3f}, RMSE: {baseline_result['rmse']:.3f}")
print(format_benchmark_label(baseline_model["baselines"]))
display(baseline_benchmarks)
display(baseline_result["summary"])
print("Max absolute level accounting identity error:", baseline_contributions["identity_check"].abs().max())


def estimate_coefficient_sample_models(
    data: pd.DataFrame,
    sample_windows: list,
    full_sample_model: dict,
) -> tuple[dict, pd.DataFrame]:
    models = {}
    rows = []
    for spec in sample_windows:
        sample_label = spec["sample"]
        sample_start = spec.get("start")
        sample_end = spec.get("end")

        if sample_start is None and sample_end is None:
            model = full_sample_model
        else:
            model = estimate_lag_window_model(
                data,
                demand_min_lag=DEMAND_MIN_LAG,
                demand_max_lag=DEMAND_MAX_LAG,
                supply_min_lag=SUPPLY_MIN_LAG,
                supply_max_lag=SUPPLY_MAX_LAG,
                baseline_mode=BASELINE_MODE,
                hac_lags=HAC_LAGS,
                sample_start=sample_start,
                sample_end=sample_end,
                sample_label=sample_label,
            )

        models[sample_label] = model
        result = model["result"]
        summary = result["summary"].copy()
        summary.insert(0, "term", summary.index)
        summary = summary.reset_index(drop=True)
        summary.insert(0, "sample", sample_label)
        summary.insert(1, "requested_start", "first_available" if sample_start is None else sample_start)
        summary.insert(2, "requested_end", "latest_available" if sample_end is None else sample_end)
        summary.insert(3, "sample_start", str(model["model_frame"].index.min()))
        summary.insert(4, "sample_end", str(model["model_frame"].index.max()))
        summary.insert(5, "nobs", int(result["nobs"]))
        summary.insert(6, "r2", float(result["r2"]))
        summary.insert(7, "rmse", float(result["rmse"]))
        rows.append(summary)

    return models, pd.concat(rows, ignore_index=True)


coefficient_sample_models, coefficient_sample_table = estimate_coefficient_sample_models(
    raw,
    COEFFICIENT_SAMPLE_WINDOWS,
    full_sample_model=baseline_model,
)

coefficient_sample_metrics = (
    coefficient_sample_table[
        ["sample", "requested_start", "requested_end", "sample_start", "sample_end", "nobs", "r2", "rmse"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
coefficient_term_order = ["const", "E_c", "D_c", "S_c", "lag_pi_c"]
coefficient_sample_coefficients_wide = (
    coefficient_sample_table.pivot(index="term", columns="sample", values="coef")
    .reindex(coefficient_term_order)
    [[spec["sample"] for spec in COEFFICIENT_SAMPLE_WINDOWS]]
)

print("\nCoefficient sample windows")
display(coefficient_sample_metrics.round(3))
print("Coefficient estimates by sample")
display(
    coefficient_sample_table[
        ["sample", "term", "coef", "newey_west_se", "t_stat", "p_value_normal_approx", "nobs", "r2", "rmse"]
    ].round(4)
)
print("Coefficient estimates, wide view")
display(coefficient_sample_coefficients_wide.round(4))

## Variance Decomposition Based on R-Squared

Following Section 5 of the reference paper, the fitted regression is treated as an in-sample identity after adding the residual term. For this full-level notebook, the dependent variable is the 12-month core PCE inflation rate, `pi_12m`.

After expressing each variable as a deviation from its sample mean, each source's variance share is:

`share_j = 100 * beta_j * Cov(x_j, pi_12m) / Var(pi_12m)`

The residual share is `100 * (1 - R²)`. The source shares can fall outside the 0% to 100% range because the explanatory variables are not orthogonal to each other. The rows below report the decomposition for the full sample and for the two 2020 split samples using each sample's own estimated coefficients.

In [ ]:
VARIANCE_DECOMPOSITION_TERMS = [
    ("Demand", "D_c"),
    ("Supply", "S_c"),
    ("Expected inflation", "E_c"),
    ("Lagged inflation", "lag_pi_c"),
]


def variance_decomposition_from_model(model: dict, sample_label: str) -> pd.DataFrame:
    result = model["result"]
    model_frame = model["model_frame"].copy()
    y_col = result["y_name"]
    y = model_frame[y_col].astype(float)
    y_centered = y - y.mean()
    y_ss = float((y_centered ** 2).sum())
    if y_ss <= 0:
        raise ValueError(f"Dependent variable has zero variance for sample {sample_label}.")

    rows = []
    for source, x_col in VARIANCE_DECOMPOSITION_TERMS:
        x = model_frame[x_col].astype(float)
        x_centered = x - x.mean()
        covariance_numerator = float((x_centered * y_centered).sum())
        covariance = covariance_numerator / len(model_frame)
        coefficient = float(result["params"][x_col])
        share = 100.0 * coefficient * covariance_numerator / y_ss
        rows.append({
            "sample": sample_label,
            "source": source,
            "regressor": x_col,
            "sample_start": str(model_frame.index.min()),
            "sample_end": str(model_frame.index.max()),
            "nobs": int(result["nobs"]),
            "r2_percent": 100.0 * float(result["r2"]),
            "coefficient": coefficient,
            "covariance_with_inflation": covariance,
            "variance_share_percent": share,
            "variance_share_from_cov_percent": share,
        })

    residual = result["residuals"].loc[model_frame.index].astype(float)
    residual_centered = residual - residual.mean()
    residual_covariance_numerator = float((residual_centered * y_centered).sum())
    residual_covariance = residual_covariance_numerator / len(model_frame)
    residual_share_from_cov = 100.0 * residual_covariance_numerator / y_ss
    residual_share_from_r2 = 100.0 * (1.0 - float(result["r2"]))
    rows.append({
        "sample": sample_label,
        "source": "Residual",
        "regressor": "residual",
        "sample_start": str(model_frame.index.min()),
        "sample_end": str(model_frame.index.max()),
        "nobs": int(result["nobs"]),
        "r2_percent": 100.0 * float(result["r2"]),
        "coefficient": 1.0,
        "covariance_with_inflation": residual_covariance,
        "variance_share_percent": residual_share_from_r2,
        "variance_share_from_cov_percent": residual_share_from_cov,
    })

    return pd.DataFrame(rows)


variance_decomposition_table = pd.concat(
    [
        variance_decomposition_from_model(model, sample_label)
        for sample_label, model in coefficient_sample_models.items()
    ],
    ignore_index=True,
)

variance_source_order = ["Demand", "Supply", "Expected inflation", "Lagged inflation", "Residual"]
variance_sample_order = [spec["sample"] for spec in COEFFICIENT_SAMPLE_WINDOWS]
variance_decomposition_table["sample"] = pd.Categorical(
    variance_decomposition_table["sample"],
    categories=variance_sample_order,
    ordered=True,
)
variance_decomposition_table["source"] = pd.Categorical(
    variance_decomposition_table["source"],
    categories=variance_source_order,
    ordered=True,
)
variance_decomposition_table = variance_decomposition_table.sort_values(["sample", "source"]).reset_index(drop=True)

variance_decomposition_wide = (
    variance_decomposition_table.pivot(index="source", columns="sample", values="variance_share_percent")
    .reindex(variance_source_order)
    [[spec["sample"] for spec in COEFFICIENT_SAMPLE_WINDOWS]]
)

variance_decomposition_checks = []
for sample, group in variance_decomposition_table.groupby("sample", observed=False):
    explained_share = float(group.loc[group["source"] != "Residual", "variance_share_percent"].sum())
    residual_share = float(group.loc[group["source"] == "Residual", "variance_share_percent"].iloc[0])
    residual_share_from_cov = float(group.loc[group["source"] == "Residual", "variance_share_from_cov_percent"].iloc[0])
    r2_percent = float(group["r2_percent"].iloc[0])
    variance_decomposition_checks.append({
        "sample": sample,
        "explained_share_percent": explained_share,
        "r2_percent": r2_percent,
        "explained_minus_r2": explained_share - r2_percent,
        "residual_share_percent": residual_share,
        "residual_share_from_cov_percent": residual_share_from_cov,
        "residual_minus_100_minus_r2": residual_share - (100.0 - r2_percent),
        "total_share_percent": explained_share + residual_share,
        "total_minus_100": explained_share + residual_share - 100.0,
    })
variance_decomposition_checks = pd.DataFrame(variance_decomposition_checks)

print("Variance decomposition shares by source")
display(
    variance_decomposition_table[
        [
            "sample",
            "source",
            "variance_share_percent",
            "coefficient",
            "covariance_with_inflation",
            "nobs",
            "r2_percent",
        ]
    ].round(3)
)

print("Variance decomposition shares, wide view")
display(variance_decomposition_wide.round(3))

print("Variance decomposition accounting checks")
display(variance_decomposition_checks.round(6))

## Main Two-Panel Full-Level Decomposition Chart

Panel A shows actual and fitted 12-month core PCE inflation in levels. Panel B keeps the chart in inflation-rate levels, but treats the model constant as a reference line instead of a filled area.

The moving components are stacked as signed contributions around the model baseline:

`actual inflation = baseline + expectations + demand + supply + lagged inflation + residual`

Positive components stack upward from the baseline and negative components stack downward from the baseline. The overlaid actual line is the algebraic signed sum of all components.

In [ ]:
FULL_COMPONENT_ORDER = ["baseline", "expectations", "demand", "supply", "lagged_inflation", "residual"]
DYNAMIC_COMPONENT_ORDER = ["expectations", "demand", "supply", "lagged_inflation", "residual"]
FITTED_COMPONENT_ORDER = ["baseline", "expectations", "demand", "supply", "lagged_inflation"]
FITTED_DYNAMIC_COMPONENT_ORDER = ["expectations", "demand", "supply", "lagged_inflation"]
LEVEL_COMPONENT_LABELS = {
    "baseline": "Constant / baseline",
    "expectations": "Expectations",
    "demand": "Demand index",
    "supply": "Supply index",
    "lagged_inflation": "Lagged inflation",
    "residual": "Residual / other",
}
LEVEL_COMPONENT_COLORS = {
    "baseline": "#6B7280",
    "expectations": "#2F6FED",
    "demand": "#269C72",
    "supply": "#C75B39",
    "lagged_inflation": "#7C5CC4",
    "residual": "#D0A33A",
}


def make_level_contributions(contributions: pd.DataFrame) -> pd.DataFrame:
    out = contributions.copy()
    out["fitted_dynamic_sum"] = out[FITTED_DYNAMIC_COMPONENT_ORDER].sum(axis=1)
    out["actual_dynamic_sum"] = out[DYNAMIC_COMPONENT_ORDER].sum(axis=1)
    out["fitted_component_sum"] = out["baseline"] + out["fitted_dynamic_sum"]
    out["actual_component_sum"] = out["baseline"] + out["actual_dynamic_sum"]
    out["fitted_minus_baseline"] = out["fitted"] - out["baseline"]
    out["actual_minus_baseline"] = out["actual"] - out["baseline"]
    out["fitted_identity_check"] = out["fitted_component_sum"] - out["fitted"]
    out["actual_identity_check"] = out["actual_component_sum"] - out["actual"]
    return out


def plot_signed_component_bars(ax, x_num, plot_frame: pd.DataFrame, component_order: list):
    positive_base = plot_frame["baseline"].astype(float).to_numpy().copy()
    negative_base = plot_frame["baseline"].astype(float).to_numpy().copy()
    bar_width_days = 24

    for component in component_order:
        y = plot_frame[component].astype(float).to_numpy()
        y_pos = np.where(y > 0, y, 0.0)
        y_neg = np.where(y < 0, y, 0.0)
        color = LEVEL_COMPONENT_COLORS[component]
        label = LEVEL_COMPONENT_LABELS[component]

        ax.bar(
            x_num,
            y_pos,
            bottom=positive_base,
            width=bar_width_days,
            color=color,
            alpha=0.72,
            linewidth=0,
            align="center",
            label=label,
        )
        ax.bar(
            x_num,
            y_neg,
            bottom=negative_base,
            width=bar_width_days,
            color=color,
            alpha=0.72,
            linewidth=0,
            align="center",
        )
        positive_base += y_pos
        negative_base += y_neg

    return positive_base, negative_base


def plot_two_panel_full_decomposition(
    contributions: pd.DataFrame,
    result: dict,
    baselines: dict,
    settings: dict,
):
    levels = make_level_contributions(contributions).dropna(subset=["actual"])
    x = levels.index.to_timestamp(how="end")
    x_num = mdates.date2num(x.to_pydatetime())
    baseline_level = float(result["params"]["const"])

    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(14, 10),
        sharex=True,
        gridspec_kw={"height_ratios": [1.0, 1.45], "hspace": 0.12},
    )
    ax_level, ax_stack = axes

    ax_level.plot(x_num, levels["actual"], color="#111827", linewidth=2.1, label="Actual 12-month core PCE")
    ax_level.plot(x_num, levels["fitted"], color="#2563EB", linewidth=1.5, linestyle="--", label="Fitted excluding residual")
    ax_level.axhline(baseline_level, color=LEVEL_COMPONENT_COLORS["baseline"], linestyle="-.", linewidth=1.2, label="Constant / baseline")
    ax_level.set_ylabel("Percent")
    ax_level.set_title("Core PCE Inflation: EXPINF1YR Full-Level NKPC Decomposition, v2", loc="left", fontsize=15, fontweight="bold", pad=12)

    subtitle = (
        f"{format_benchmark_label(baselines)}; "
        f"expectations: EXPINF1YR; "
        f"demand/supply: Demand_Supply_Data.xlsx; "
        f"demand lags: {settings['demand_min_lag']}-{settings['demand_max_lag']}m; "
        f"supply lags: {settings['supply_min_lag']}-{settings['supply_max_lag']}m; "
        f"HAC lags: {settings['hac_lags']}; full-sample coefficients"
    )
    ax_level.text(0, 1.02, subtitle, transform=ax_level.transAxes, fontsize=9.5, color="#4B5563")

    plot_signed_component_bars(ax_stack, x_num, levels, DYNAMIC_COMPONENT_ORDER)
    ax_stack.axhline(baseline_level, color=LEVEL_COMPONENT_COLORS["baseline"], linestyle="-.", linewidth=1.3, label="Constant / baseline")
    ax_stack.plot(x_num, levels["actual"], color="#111827", linewidth=2.1, label="Actual = signed sum")
    ax_stack.plot(x_num, levels["fitted"], color="#111827", linewidth=1.25, linestyle="--", label="Fitted = sum excluding residual")
    ax_stack.set_ylabel("Percent")
    ax_stack.xaxis_date()
    ax_stack.xaxis.set_major_locator(mdates.YearLocator(2))
    ax_stack.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    handles_level, labels_level = ax_level.get_legend_handles_labels()
    handles_stack, labels_stack = ax_stack.get_legend_handles_labels()
    handles = handles_level + handles_stack
    labels = labels_level + labels_stack
    dedup = dict(zip(labels, handles))
    fig.legend(
        dedup.values(),
        dedup.keys(),
        ncol=4,
        frameon=False,
        loc="lower left",
        bbox_to_anchor=(0.055, 0.082),
    )

    footnote = (
        "Panel B is a signed full-level contribution chart. The constant is shown as the model baseline, not as a filled driver. "
        "Positive components stack upward from that baseline and negative components stack downward; the black actual line is the algebraic signed sum. "
        "Thus baseline plus expectations, demand, supply, lagged inflation, and residual equals actual 12-month core PCE inflation. "
        "Expectations are FRED EXPINF1YR; demand and supply are from Demand_Supply_Data.xlsx. "
        "The decomposition is regression-implied and should not be interpreted as structural causal shares."
    )
    fig.text(0.055, 0.018, textwrap.fill(footnote, 165), fontsize=8.5, color="#4B5563")
    fig.subplots_adjust(left=0.07, right=0.99, top=0.92, bottom=0.23)
    return fig, axes, levels


main_fig, main_axes, baseline_level_contributions = plot_two_panel_full_decomposition(
    baseline_contributions,
    baseline_result,
    baseline_model["baselines"],
    baseline_model["settings"],
)
plt.show()

latest = baseline_level_contributions.iloc[-1]
print("Max absolute actual level identity error:", baseline_level_contributions["actual_identity_check"].abs().max())
print("Max absolute fitted level identity error:", baseline_level_contributions["fitted_identity_check"].abs().max())
print(
    f"Latest observation {baseline_level_contributions.index[-1]}: "
    f"baseline + expectations + demand + supply + lagged inflation + residual = "
    f"{latest['actual_component_sum']:.3f}%, actual = {latest['actual']:.3f}%"
)
print(f"Model baseline / constant: {baseline_result['params']['const']:.3f}%")

## 12-Month Change in Inflation Decomposition

This section follows the dependent-variable setup used in the reference paper's variance-decomposition tables. Instead of decomposing the level of 12-month core PCE inflation, it decomposes the 12-month change in that inflation rate:

`delta_pi_12m = pi_12m - pi_12m.shift(12)`

The regressors are also expressed as 12-month changes: expected inflation, the lag-window demand proxy, the lag-window supply proxy, and lagged 12-month inflation. The fitted identity is:

`delta_pi_12m = baseline + expectations change + demand change + supply change + lagged inflation change + residual`

As in the level decomposition, the regressors are centered on each estimation sample's own means so the intercept is the model baseline for that sample.

In [ ]:
CHANGE_COMPONENT_LABELS = {
    "baseline": "Constant / baseline",
    "expectations": "Expectations change",
    "demand": "Demand change",
    "supply": "Supply change",
    "lagged_inflation": "Lagged inflation change",
    "residual": "Residual / other",
}


def change_demand_base(frame: pd.DataFrame) -> pd.Series:
    if "DEMAND_TRANSFORM" in globals():
        if DEMAND_TRANSFORM == "log_vu":
            return frame["log_vu"]
        if DEMAND_TRANSFORM == "raw_vu":
            return frame["vu"]
        raise ValueError("DEMAND_TRANSFORM must be 'log_vu' or 'raw_vu'.")
    return frame["demand"]


def change_supply_base(frame: pd.DataFrame) -> pd.Series:
    if "gscpi" in frame.columns:
        return frame["gscpi"]
    return frame["supply"]


def build_change_contributions(result: dict, model_frame: pd.DataFrame) -> pd.DataFrame:
    contributions = pd.DataFrame(index=model_frame.index)
    contributions["baseline"] = result["params"]["const"]
    contributions["expectations"] = result["params"]["delta_E_c"] * model_frame["delta_E_c"]
    contributions["demand"] = result["params"]["delta_D_c"] * model_frame["delta_D_c"]
    contributions["supply"] = result["params"]["delta_S_c"] * model_frame["delta_S_c"]
    contributions["lagged_inflation"] = result["params"]["delta_lag_pi_c"] * model_frame["delta_lag_pi_c"]
    contributions["fitted"] = result["fitted"]
    contributions["actual"] = model_frame["delta_pi_12m"]
    contributions["residual"] = contributions["actual"] - contributions["fitted"]
    contributions["identity_check"] = (
        contributions[["baseline", "expectations", "demand", "supply", "lagged_inflation", "residual"]].sum(axis=1)
        - contributions["actual"]
    )
    return contributions


def estimate_change_12m_model(
    data: pd.DataFrame,
    demand_min_lag: int = DEMAND_MIN_LAG,
    demand_max_lag: int = DEMAND_MAX_LAG,
    supply_min_lag: int = SUPPLY_MIN_LAG,
    supply_max_lag: int = SUPPLY_MAX_LAG,
    hac_lags: int = HAC_LAGS,
    sample_start: str | None = None,
    sample_end: str | None = None,
    sample_label: str = "full_sample",
) -> dict:
    frame = data.copy()
    frame["change_demand_base"] = change_demand_base(frame)
    frame["change_supply_base"] = change_supply_base(frame)
    frame["Dbar_change_level"] = lag_window_average(frame["change_demand_base"], demand_min_lag, demand_max_lag)
    frame["Sbar_change_level"] = lag_window_average(frame["change_supply_base"], supply_min_lag, supply_max_lag)

    frame["delta_pi_12m"] = frame["pi_12m"] - frame["pi_12m"].shift(12)
    frame["delta_E"] = frame["exp_1yr"] - frame["exp_1yr"].shift(12)
    frame["delta_D"] = frame["Dbar_change_level"] - frame["Dbar_change_level"].shift(12)
    frame["delta_S"] = frame["Sbar_change_level"] - frame["Sbar_change_level"].shift(12)
    frame["delta_lag_pi"] = frame["lag_pi_12m"] - frame["lag_pi_12m"].shift(12)

    required = ["delta_pi_12m", "delta_E", "delta_D", "delta_S", "delta_lag_pi"]
    preliminary_sample = frame[required].dropna()
    sample_index = filter_period_index(preliminary_sample.index, sample_start, sample_end)
    if len(sample_index) == 0:
        raise ValueError(f"No usable observations for 12-month-change sample {sample_label}: {sample_start} to {sample_end}.")

    sample = frame.loc[sample_index, required]
    centers = {
        "delta_E_star": sample["delta_E"].mean(),
        "delta_D_star": sample["delta_D"].mean(),
        "delta_S_star": sample["delta_S"].mean(),
        "delta_lag_pi_star": sample["delta_lag_pi"].mean(),
    }
    frame["delta_E_c"] = frame["delta_E"] - centers["delta_E_star"]
    frame["delta_D_c"] = frame["delta_D"] - centers["delta_D_star"]
    frame["delta_S_c"] = frame["delta_S"] - centers["delta_S_star"]
    frame["delta_lag_pi_c"] = frame["delta_lag_pi"] - centers["delta_lag_pi_star"]

    model_cols = ["delta_E_c", "delta_D_c", "delta_S_c", "delta_lag_pi_c"]
    model_frame = frame.loc[sample_index, ["delta_pi_12m"] + model_cols].dropna()
    result = fit_ols_hac(model_frame, "delta_pi_12m", model_cols, hac_lags=hac_lags)
    contributions = build_change_contributions(result, model_frame)

    return {
        "frame": frame,
        "model_frame": model_frame,
        "result": result,
        "contributions": contributions,
        "centers": centers,
        "settings": {
            "dependent_variable": "delta_pi_12m = pi_12m - pi_12m.shift(12)",
            "sample_label": sample_label,
            "sample_start": sample_start,
            "sample_end": sample_end,
            "demand_min_lag": demand_min_lag,
            "demand_max_lag": demand_max_lag,
            "supply_min_lag": supply_min_lag,
            "supply_max_lag": supply_max_lag,
            "hac_lags": hac_lags,
        },
    }


def estimate_change_sample_models(data: pd.DataFrame, sample_windows: list) -> tuple[dict, pd.DataFrame]:
    models = {}
    rows = []
    for spec in sample_windows:
        sample_label = spec["sample"]
        model = estimate_change_12m_model(
            data,
            demand_min_lag=DEMAND_MIN_LAG,
            demand_max_lag=DEMAND_MAX_LAG,
            supply_min_lag=SUPPLY_MIN_LAG,
            supply_max_lag=SUPPLY_MAX_LAG,
            hac_lags=HAC_LAGS,
            sample_start=spec.get("start"),
            sample_end=spec.get("end"),
            sample_label=sample_label,
        )
        models[sample_label] = model
        result = model["result"]
        summary = result["summary"].copy()
        summary.insert(0, "term", summary.index)
        summary = summary.reset_index(drop=True)
        summary.insert(0, "sample", sample_label)
        summary.insert(1, "requested_start", "first_available" if spec.get("start") is None else spec.get("start"))
        summary.insert(2, "requested_end", "latest_available" if spec.get("end") is None else spec.get("end"))
        summary.insert(3, "sample_start", str(model["model_frame"].index.min()))
        summary.insert(4, "sample_end", str(model["model_frame"].index.max()))
        summary.insert(5, "nobs", int(result["nobs"]))
        summary.insert(6, "r2", float(result["r2"]))
        summary.insert(7, "rmse", float(result["rmse"]))
        rows.append(summary)
    return models, pd.concat(rows, ignore_index=True)


change_sample_models, change_coefficient_sample_table = estimate_change_sample_models(raw, COEFFICIENT_SAMPLE_WINDOWS)
change_baseline_model = change_sample_models["full_sample"]
change_baseline_result = change_baseline_model["result"]
change_contributions = change_baseline_model["contributions"]

change_sample_metrics = (
    change_coefficient_sample_table[
        ["sample", "requested_start", "requested_end", "sample_start", "sample_end", "nobs", "r2", "rmse"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

change_term_order = ["const", "delta_E_c", "delta_D_c", "delta_S_c", "delta_lag_pi_c"]
change_coefficients_wide = (
    change_coefficient_sample_table.pivot(index="term", columns="sample", values="coef")
    .reindex(change_term_order)
    [[spec["sample"] for spec in COEFFICIENT_SAMPLE_WINDOWS]]
)

print("12-month-change regression sample windows")
display(change_sample_metrics.round(3))
print("12-month-change coefficient estimates, wide view")
display(change_coefficients_wide.round(4))
print("Full-sample 12-month-change coefficient table")
display(change_baseline_result["summary"].round(4))
print("Max absolute 12-month-change accounting identity error:", change_contributions["identity_check"].abs().max())


CHANGE_VARIANCE_TERMS = [
    ("Demand", "delta_D_c"),
    ("Supply", "delta_S_c"),
    ("Expected inflation", "delta_E_c"),
    ("Lagged inflation", "delta_lag_pi_c"),
]


def change_variance_decomposition_from_model(model: dict, sample_label: str) -> pd.DataFrame:
    result = model["result"]
    model_frame = model["model_frame"].copy()
    y = model_frame[result["y_name"]].astype(float)
    y_centered = y - y.mean()
    y_ss = float((y_centered ** 2).sum())
    if y_ss <= 0:
        raise ValueError(f"Dependent variable has zero variance for 12-month-change sample {sample_label}.")

    rows = []
    for source, x_col in CHANGE_VARIANCE_TERMS:
        x = model_frame[x_col].astype(float)
        x_centered = x - x.mean()
        covariance_numerator = float((x_centered * y_centered).sum())
        coefficient = float(result["params"][x_col])
        rows.append({
            "sample": sample_label,
            "source": source,
            "regressor": x_col,
            "sample_start": str(model_frame.index.min()),
            "sample_end": str(model_frame.index.max()),
            "nobs": int(result["nobs"]),
            "r2_percent": 100.0 * float(result["r2"]),
            "coefficient": coefficient,
            "covariance_with_delta_inflation": covariance_numerator / len(model_frame),
            "variance_share_percent": 100.0 * coefficient * covariance_numerator / y_ss,
        })

    residual = result["residuals"].loc[model_frame.index].astype(float)
    residual_centered = residual - residual.mean()
    residual_covariance_numerator = float((residual_centered * y_centered).sum())
    rows.append({
        "sample": sample_label,
        "source": "Residual",
        "regressor": "residual",
        "sample_start": str(model_frame.index.min()),
        "sample_end": str(model_frame.index.max()),
        "nobs": int(result["nobs"]),
        "r2_percent": 100.0 * float(result["r2"]),
        "coefficient": 1.0,
        "covariance_with_delta_inflation": residual_covariance_numerator / len(model_frame),
        "variance_share_percent": 100.0 * (1.0 - float(result["r2"])),
    })
    return pd.DataFrame(rows)


change_variance_decomposition_table = pd.concat(
    [
        change_variance_decomposition_from_model(model, sample_label)
        for sample_label, model in change_sample_models.items()
    ],
    ignore_index=True,
)

change_variance_source_order = ["Demand", "Supply", "Expected inflation", "Lagged inflation", "Residual"]
change_variance_sample_order = [spec["sample"] for spec in COEFFICIENT_SAMPLE_WINDOWS]
change_variance_decomposition_table["sample"] = pd.Categorical(
    change_variance_decomposition_table["sample"],
    categories=change_variance_sample_order,
    ordered=True,
)
change_variance_decomposition_table["source"] = pd.Categorical(
    change_variance_decomposition_table["source"],
    categories=change_variance_source_order,
    ordered=True,
)
change_variance_decomposition_table = change_variance_decomposition_table.sort_values(["sample", "source"]).reset_index(drop=True)

change_variance_decomposition_wide = (
    change_variance_decomposition_table.pivot(index="source", columns="sample", values="variance_share_percent")
    .reindex(change_variance_source_order)
    [[spec["sample"] for spec in COEFFICIENT_SAMPLE_WINDOWS]]
)

change_variance_decomposition_checks = []
for sample, group in change_variance_decomposition_table.groupby("sample", observed=False):
    explained_share = float(group.loc[group["source"] != "Residual", "variance_share_percent"].sum())
    residual_share = float(group.loc[group["source"] == "Residual", "variance_share_percent"].iloc[0])
    r2_percent = float(group["r2_percent"].iloc[0])
    change_variance_decomposition_checks.append({
        "sample": sample,
        "explained_share_percent": explained_share,
        "r2_percent": r2_percent,
        "explained_minus_r2": explained_share - r2_percent,
        "residual_share_percent": residual_share,
        "residual_minus_100_minus_r2": residual_share - (100.0 - r2_percent),
        "total_share_percent": explained_share + residual_share,
        "total_minus_100": explained_share + residual_share - 100.0,
    })
change_variance_decomposition_checks = pd.DataFrame(change_variance_decomposition_checks)

print("12-month-change variance decomposition shares")
display(change_variance_decomposition_wide.round(3))
print("12-month-change variance decomposition checks")
display(change_variance_decomposition_checks.round(6))


def plot_signed_change_component_bars(ax, x_num, plot_frame: pd.DataFrame, component_order: list):
    positive_base = plot_frame["baseline"].astype(float).to_numpy().copy()
    negative_base = plot_frame["baseline"].astype(float).to_numpy().copy()
    bar_width_days = 24

    for component in component_order:
        y = plot_frame[component].astype(float).to_numpy()
        y_pos = np.where(y > 0, y, 0.0)
        y_neg = np.where(y < 0, y, 0.0)
        color = LEVEL_COMPONENT_COLORS[component]
        label = CHANGE_COMPONENT_LABELS[component]

        ax.bar(x_num, y_pos, bottom=positive_base, width=bar_width_days, color=color, alpha=0.72, linewidth=0, align="center", label=label)
        ax.bar(x_num, y_neg, bottom=negative_base, width=bar_width_days, color=color, alpha=0.72, linewidth=0, align="center")
        positive_base += y_pos
        negative_base += y_neg


def plot_change_12m_decomposition(contributions: pd.DataFrame, result: dict):
    levels = make_level_contributions(contributions).dropna(subset=["actual"])
    x = levels.index.to_timestamp(how="end")
    x_num = mdates.date2num(x.to_pydatetime())
    baseline_level = float(result["params"]["const"])

    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(14, 9.5),
        sharex=True,
        gridspec_kw={"height_ratios": [1.0, 1.45], "hspace": 0.12},
    )
    ax_level, ax_stack = axes

    ax_level.plot(x_num, levels["actual"], color="#111827", linewidth=2.1, label="Actual 12-month change")
    ax_level.plot(x_num, levels["fitted"], color="#2563EB", linewidth=1.5, linestyle="--", label="Fitted excluding residual")
    ax_level.axhline(baseline_level, color=LEVEL_COMPONENT_COLORS["baseline"], linestyle="-.", linewidth=1.2, label="Constant / baseline")
    ax_level.axhline(0, color="#9CA3AF", linewidth=0.8)
    ax_level.set_ylabel("Percentage points")
    ax_level.set_title("12-Month Change in Core PCE Inflation: NKPC Decomposition", loc="left", fontsize=15, fontweight="bold", pad=12)
    ax_level.text(
        0,
        1.02,
        "Dependent variable: pi_12m - pi_12m.shift(12); regressors are 12-month changes in expectations, demand, supply, and lagged inflation.",
        transform=ax_level.transAxes,
        fontsize=9.5,
        color="#4B5563",
    )

    plot_signed_change_component_bars(ax_stack, x_num, levels, DYNAMIC_COMPONENT_ORDER)
    ax_stack.axhline(baseline_level, color=LEVEL_COMPONENT_COLORS["baseline"], linestyle="-.", linewidth=1.3, label="Constant / baseline")
    ax_stack.axhline(0, color="#9CA3AF", linewidth=0.8)
    ax_stack.plot(x_num, levels["actual"], color="#111827", linewidth=2.1, label="Actual = signed sum")
    ax_stack.plot(x_num, levels["fitted"], color="#111827", linewidth=1.25, linestyle="--", label="Fitted = sum excluding residual")
    ax_stack.set_ylabel("Percentage points")
    ax_stack.xaxis_date()
    ax_stack.xaxis.set_major_locator(mdates.YearLocator(2))
    ax_stack.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    handles_level, labels_level = ax_level.get_legend_handles_labels()
    handles_stack, labels_stack = ax_stack.get_legend_handles_labels()
    dedup = dict(zip(labels_level + labels_stack, handles_level + handles_stack))
    fig.legend(dedup.values(), dedup.keys(), ncol=4, frameon=False, loc="lower left", bbox_to_anchor=(0.055, 0.08))
    footnote = (
        "This chart decomposes changes in the 12-month inflation rate, not the inflation-rate level. "
        "The residual share in the accompanying variance decomposition is 100 * (1 - R-squared). "
        "Shares can fall outside 0-100 because the sources are not orthogonal."
    )
    fig.text(0.055, 0.018, textwrap.fill(footnote, 165), fontsize=8.5, color="#4B5563")
    fig.subplots_adjust(left=0.07, right=0.99, top=0.91, bottom=0.22)
    return fig, axes, levels


change_fig, change_axes, change_level_contributions = plot_change_12m_decomposition(
    change_contributions,
    change_baseline_result,
)
plt.show()

latest_change = change_level_contributions.iloc[-1]
print("Max absolute 12-month-change actual identity error:", change_level_contributions["actual_identity_check"].abs().max())
print(
    f"Latest observation {change_level_contributions.index[-1]}: "
    f"component sum = {latest_change['actual_component_sum']:.3f} percentage points, "
    f"actual 12-month change = {latest_change['actual']:.3f} percentage points"
)

## Diagnostics

In [ ]:
def coefficient_sign_table(result: dict) -> pd.DataFrame:
    expected_positive = {
        "E_c": "Higher expected inflation should raise inflation",
        "D_c": "Higher demand pressure should raise inflation",
        "S_c": "Higher supply pressure should raise inflation",
        "lag_pi_c": "Higher lagged inflation should imply persistence",
    }
    rows = []
    for name, interpretation in expected_positive.items():
        if name in result["params"].index:
            coef = result["params"][name]
            rows.append({
                "term": name,
                "coef": coef,
                "expected_sign": "+",
                "matches_expected": coef > 0,
                "note": interpretation,
            })
    return pd.DataFrame(rows)


def residual_diagnostics(result: dict, contributions: pd.DataFrame) -> dict:
    residual = result["residuals"]
    stats = pd.Series(
        {
            "mean": residual.mean(),
            "standard_deviation": residual.std(),
            "maximum_absolute_residual": residual.abs().max(),
            "autocorrelation_lag_1": residual.autocorr(lag=1),
            "autocorrelation_lag_3": residual.autocorr(lag=3),
            "autocorrelation_lag_6": residual.autocorr(lag=6),
            "autocorrelation_lag_12": residual.autocorr(lag=12),
        },
        name="residual_statistic",
    )
    top_residuals = contributions.assign(abs_residual=contributions["residual"].abs()).sort_values(
        "abs_residual", ascending=False
    ).head(12)[["actual", "fitted", "residual", "abs_residual"]]
    return {"stats": stats, "top_residuals": top_residuals}


diagnostic_frame = baseline_model["frame"].loc[baseline_contributions.index].copy()
diagnostic_columns = ["exp_1yr", "demand", "supply", "lag_pi_12m", "pi_12m"]

print("Coefficient sign check")
display(coefficient_sign_table(baseline_result))

print("Correlation matrix")
display(diagnostic_frame[diagnostic_columns].corr().round(3))

resid_diag = residual_diagnostics(baseline_result, baseline_contributions)
print("Residual diagnostics")
display(resid_diag["stats"].round(3).to_frame())

print("Largest absolute residual episodes")
display(resid_diag["top_residuals"].round(3))

## Robustness: Demand/Supply Lag Windows

In [ ]:
def period_window_mean(contributions: pd.DataFrame, col: str, start: str, end: str) -> float:
    window = contributions.loc[pre_pandemic_mask(contributions.index, start, end), col]
    return float(window.mean()) if len(window) else np.nan


robust_rows = []
for lag_spec in ROBUST_LAG_WINDOWS:
    model = estimate_lag_window_model(
        raw,
        demand_min_lag=lag_spec["demand"][0],
        demand_max_lag=lag_spec["demand"][1],
        supply_min_lag=lag_spec["supply"][0],
        supply_max_lag=lag_spec["supply"][1],
        baseline_mode=BASELINE_MODE,
        hac_lags=HAC_LAGS,
    )
    result = model["result"]
    contributions = model["contributions"]
    robust_rows.append({
        "model": lag_spec["name"],
        "demand_lags": f"{lag_spec['demand'][0]}-{lag_spec['demand'][1]}",
        "supply_lags": f"{lag_spec['supply'][0]}-{lag_spec['supply'][1]}",
        "nobs": result["nobs"],
        "r2": result["r2"],
        "rmse": result["rmse"],
        "coef_expectations": result["params"].get("E_c", np.nan),
        "coef_demand": result["params"].get("D_c", np.nan),
        "coef_supply": result["params"].get("S_c", np.nan),
        "coef_lagged_inflation": result["params"].get("lag_pi_c", np.nan),
        "latest_actual": contributions["actual"].iloc[-1],
        "latest_fitted": contributions["fitted"].iloc[-1],
        "latest_demand": contributions["demand"].iloc[-1],
        "latest_supply": contributions["supply"].iloc[-1],
        "avg_demand_2021_2023": period_window_mean(contributions, "demand", "2021-01", "2023-12"),
        "avg_supply_2021_2023": period_window_mean(contributions, "supply", "2021-01", "2023-12"),
        "avg_demand_2024_2026": period_window_mean(contributions, "demand", "2024-01", "2026-12"),
        "avg_supply_2024_2026": period_window_mean(contributions, "supply", "2024-01", "2026-12"),
    })

robustness_table = pd.DataFrame(robust_rows)
display(robustness_table.round(3))

## Distributed-Lag Robustness

This specification estimates separate demand and supply coefficients at selected lag months and groups those lagged terms into total demand and supply contributions. Under `sample_mean` centering, each lagged column is centered on its own estimation-sample mean.

In [ ]:
def distributed_lag_stars(
    model_frame: pd.DataFrame,
    d_lag_cols: list,
    s_lag_cols: list,
    baseline_mode: str = BASELINE_MODE,
) -> dict:
    if baseline_mode == "sample_mean":
        stars = {"E_star": model_frame["exp_1yr"].mean(), "PI_star": model_frame["lag_pi_12m"].mean(), "mode": "sample_mean"}
        for col in d_lag_cols + s_lag_cols:
            stars[f"{col}_star"] = model_frame[col].mean()
        return stars

    window = model_frame.loc[pre_pandemic_mask(model_frame.index, PRE_PANDEMIC_START, PRE_PANDEMIC_END)]
    stars = {
        "E_star": window["exp_1yr"].mean() if EXPECTATIONS_STAR_OVERRIDE is None else float(EXPECTATIONS_STAR_OVERRIDE),
        "PI_star": window["pi_12m"].mean() if PI_STAR_OVERRIDE is None else float(PI_STAR_OVERRIDE),
        "mode": "pre_pandemic_average",
        "window": f"{PRE_PANDEMIC_START} to {PRE_PANDEMIC_END}",
    }
    for col in d_lag_cols + s_lag_cols:
        stars[f"{col}_star"] = window[col].mean()
    return stars


def estimate_distributed_lag_model(data: pd.DataFrame, lag_list: list, hac_lags: int = HAC_LAGS) -> dict:
    frame = data.copy()
    d_lag_cols = []
    s_lag_cols = []
    for lag in lag_list:
        d_col = f"D_lag_{lag}"
        s_col = f"S_lag_{lag}"
        frame[d_col] = frame["demand"].shift(lag)
        frame[s_col] = frame["supply"].shift(lag)
        d_lag_cols.append(d_col)
        s_lag_cols.append(s_col)

    required = ["pi_12m", "exp_1yr", "lag_pi_12m"] + d_lag_cols + s_lag_cols
    model_frame = frame[required].dropna()
    stars = distributed_lag_stars(
        model_frame=model_frame,
        d_lag_cols=d_lag_cols,
        s_lag_cols=s_lag_cols,
        baseline_mode=BASELINE_MODE,
    )

    model_frame = model_frame.copy()
    model_frame["E_c"] = model_frame["exp_1yr"] - stars["E_star"]
    model_frame["lag_pi_c"] = model_frame["lag_pi_12m"] - stars["PI_star"]

    x_cols = ["E_c"]
    d_centered = []
    s_centered = []
    for col in d_lag_cols:
        c_col = f"{col}_c"
        model_frame[c_col] = model_frame[col] - stars[f"{col}_star"]
        x_cols.append(c_col)
        d_centered.append(c_col)
    for col in s_lag_cols:
        c_col = f"{col}_c"
        model_frame[c_col] = model_frame[col] - stars[f"{col}_star"]
        x_cols.append(c_col)
        s_centered.append(c_col)
    x_cols.append("lag_pi_c")

    regression_frame = model_frame[["pi_12m"] + x_cols].dropna()
    result = fit_ols_hac(regression_frame, "pi_12m", x_cols, hac_lags=hac_lags)

    contributions = pd.DataFrame(index=regression_frame.index)
    contributions["baseline"] = result["params"]["const"]
    contributions["expectations"] = result["params"]["E_c"] * regression_frame["E_c"]
    contributions["demand"] = sum(result["params"][col] * regression_frame[col] for col in d_centered)
    contributions["supply"] = sum(result["params"][col] * regression_frame[col] for col in s_centered)
    contributions["lagged_inflation"] = result["params"]["lag_pi_c"] * regression_frame["lag_pi_c"]
    contributions["fitted"] = result["fitted"]
    contributions["actual"] = regression_frame["pi_12m"]
    contributions["residual"] = contributions["actual"] - contributions["fitted"]
    contributions["identity_check"] = (
        contributions[["baseline", "expectations", "demand", "supply", "lagged_inflation", "residual"]].sum(axis=1)
        - contributions["actual"]
    )

    return {
        "frame": frame,
        "model_frame": regression_frame,
        "result": result,
        "contributions": contributions,
        "stars": stars,
        "d_lag_cols": d_centered,
        "s_lag_cols": s_centered,
    }


distributed_models = {}
distributed_rows = []
for name, lags in DISTRIBUTED_LAG_SETS.items():
    model = estimate_distributed_lag_model(raw, lags)
    distributed_models[name] = model
    result = model["result"]
    contributions = model["contributions"]
    distributed_rows.append({
        "model": name,
        "lags": ",".join(str(lag) for lag in lags),
        "nobs": result["nobs"],
        "r2": result["r2"],
        "rmse": result["rmse"],
        "latest_demand": contributions["demand"].iloc[-1],
        "latest_supply": contributions["supply"].iloc[-1],
        "max_identity_error": contributions["identity_check"].abs().max(),
    })

distributed_summary = pd.DataFrame(distributed_rows)
display(distributed_summary.round(3))
display(distributed_models["simple_0_3_6"]["result"]["summary"].round(4))

## Correlated Demand/Supply Proxy Robustness

In [ ]:
def orthogonalize_demand_supply(frame: pd.DataFrame, d_col: str = "Dbar", s_col: str = "Sbar") -> pd.DataFrame:
    tmp = frame[[d_col, s_col]].dropna()

    xs = np.column_stack([np.ones(len(tmp)), tmp[s_col].to_numpy()])
    beta_d_on_s = np.linalg.lstsq(xs, tmp[d_col].to_numpy(), rcond=None)[0]
    d_resid = tmp[d_col] - xs @ beta_d_on_s

    xd = np.column_stack([np.ones(len(tmp)), tmp[d_col].to_numpy()])
    beta_s_on_d = np.linalg.lstsq(xd, tmp[s_col].to_numpy(), rcond=None)[0]
    s_resid = tmp[s_col] - xd @ beta_s_on_d

    out = frame.copy()
    out.loc[tmp.index, "Dbar_ortho"] = 0.5 * tmp[d_col] + 0.5 * d_resid
    out.loc[tmp.index, "Sbar_ortho"] = 0.5 * tmp[s_col] + 0.5 * s_resid
    return out


def estimate_orthogonalized_model(base_model: dict) -> dict:
    frame = orthogonalize_demand_supply(base_model["frame"], d_col="Dbar", s_col="Sbar")
    required = ["pi_12m", "exp_1yr", "Dbar_ortho", "Sbar_ortho", "lag_pi_12m"]
    preliminary_sample = frame[required].dropna()
    baselines = choose_baselines(
        frame=frame,
        sample_index=preliminary_sample.index,
        demand_col="Dbar_ortho",
        supply_col="Sbar_ortho",
        baseline_mode=BASELINE_MODE,
    )

    frame["E_c"] = frame["exp_1yr"] - baselines["E_star"]
    frame["D_c"] = frame["Dbar_ortho"] - baselines["D_star"]
    frame["S_c"] = frame["Sbar_ortho"] - baselines["S_star"]
    frame["lag_pi_c"] = frame["lag_pi_12m"] - baselines["PI_star"]

    model_cols = ["E_c", "D_c", "S_c", "lag_pi_c"]
    model_frame = frame[["pi_12m"] + model_cols].dropna()
    result = fit_ols_hac(model_frame, "pi_12m", model_cols, hac_lags=HAC_LAGS)
    contributions = build_standard_contributions(result, model_frame)
    return {
        "frame": frame,
        "model_frame": model_frame,
        "result": result,
        "contributions": contributions,
        "baselines": baselines,
    }


orthogonal_model = estimate_orthogonalized_model(baseline_model)
orthogonal_result = orthogonal_model["result"]

common_index = baseline_contributions.index.intersection(orthogonal_model["contributions"].index)
orthogonal_comparison = pd.DataFrame({
    "baseline_demand": baseline_contributions.loc[common_index, "demand"],
    "orthogonalized_demand": orthogonal_model["contributions"].loc[common_index, "demand"],
    "baseline_supply": baseline_contributions.loc[common_index, "supply"],
    "orthogonalized_supply": orthogonal_model["contributions"].loc[common_index, "supply"],
})

proxy_corr = baseline_model["frame"][["Dbar", "Sbar"]].corr().iloc[0, 1]
ortho_proxy_corr = orthogonal_model["frame"][["Dbar_ortho", "Sbar_ortho"]].corr().iloc[0, 1]
print(f"Baseline Dbar/Sbar correlation: {proxy_corr:.3f}")
print(f"Orthogonalized Dbar/Sbar correlation: {ortho_proxy_corr:.3f}")
print(f"Orthogonalized model R-squared: {orthogonal_result['r2']:.3f}, RMSE: {orthogonal_result['rmse']:.3f}")
display(orthogonal_result["summary"].round(4))
display(orthogonal_comparison.tail(12).round(3))

## Optional Statsmodels Verification and Export

If `statsmodels` is available in the active Python environment, the next cell compares the custom OLS/HAC coefficients and standard errors with `statsmodels`. The export section writes the full level contribution table, benchmark values, model settings, and robustness summaries.

In [ ]:
try:
    import statsmodels.api as sm

    sm_y = baseline_model["model_frame"]["pi_12m"]
    sm_x = sm.add_constant(baseline_model["model_frame"][baseline_result["x_names"]])
    sm_res = sm.OLS(sm_y, sm_x).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})
    statsmodels_check = pd.DataFrame({
        "custom_coef": baseline_result["params"],
        "statsmodels_coef": sm_res.params,
        "coef_diff": baseline_result["params"] - sm_res.params,
        "custom_hac_se": baseline_result["std_errors"],
        "statsmodels_hac_se": sm_res.bse,
        "se_diff": baseline_result["std_errors"] - sm_res.bse,
    })
    display(statsmodels_check)
except ImportError:
    statsmodels_check = None
    print("statsmodels is not installed in this environment; skipping optional HAC verification.")

## Export Results

In [ ]:
def timestamped_export_path(path: Path) -> Path:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return path.with_name(f"{path.stem}_{stamp}{path.suffix}")


def safe_to_csv(frame: pd.DataFrame, path: Path, **kwargs) -> Path:
    try:
        frame.to_csv(path, **kwargs)
        return path
    except PermissionError:
        alternate = timestamped_export_path(path)
        frame.to_csv(alternate, **kwargs)
        print(f"{path.name} appears to be open or locked; saved alternate {alternate.name}")
        return alternate


def safe_write_text(text: str, path: Path, encoding: str = "utf-8") -> Path:
    try:
        path.write_text(text, encoding=encoding)
        return path
    except PermissionError:
        alternate = timestamped_export_path(path)
        alternate.write_text(text, encoding=encoding)
        print(f"{path.name} appears to be open or locked; saved alternate {alternate.name}")
        return alternate


def safe_save_figure(fig, path: Path, **kwargs) -> Path:
    try:
        fig.savefig(path, **kwargs)
        return path
    except PermissionError:
        alternate = timestamped_export_path(path)
        fig.savefig(alternate, **kwargs)
        print(f"{path.name} appears to be open or locked; saved alternate {alternate.name}")
        return alternate


contribution_export = baseline_level_contributions.copy()
contribution_export.insert(0, "month", contribution_export.index.astype(str))
change_contribution_export = change_level_contributions.copy()
change_contribution_export.insert(0, "month", change_contribution_export.index.astype(str))

contrib_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_full_contributions.csv"
chart_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_full_decomposition_chart.png"
summary_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_baseline_coefficients.csv"
variance_decomposition_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_variance_decomposition.csv"
variance_decomposition_checks_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_variance_decomposition_checks.csv"
coefficient_samples_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_coefficient_samples.csv"
baseline_values_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_baseline_values.csv"
robust_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_robustness_summary.csv"
distributed_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_distributed_lag_summary.csv"
orthogonal_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_orthogonalized_comparison.csv"
change_contrib_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_change_12m_contributions.csv"
change_chart_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_change_12m_decomposition_chart.png"
change_summary_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_change_12m_coefficients.csv"
change_coefficient_samples_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_change_12m_coefficient_samples.csv"
change_variance_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_change_12m_variance_decomposition.csv"
change_variance_checks_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_change_12m_variance_decomposition_checks.csv"
settings_path = OUTPUT_DIR / "core_pce_nkpc_expinf1yr_v2_model_settings.json"

written_paths = []
written_paths.append(safe_to_csv(contribution_export, contrib_path, index=False))
written_paths.append(safe_to_csv(change_contribution_export, change_contrib_path, index=False))
written_paths.append(safe_to_csv(baseline_result["summary"], summary_path))
written_paths.append(safe_to_csv(coefficient_sample_table, coefficient_samples_path, index=False))
written_paths.append(safe_to_csv(change_baseline_result["summary"], change_summary_path))
written_paths.append(safe_to_csv(change_coefficient_sample_table, change_coefficient_samples_path, index=False))
written_paths.append(safe_to_csv(change_variance_decomposition_table, change_variance_path, index=False))
written_paths.append(safe_to_csv(change_variance_decomposition_checks, change_variance_checks_path, index=False))
written_paths.append(safe_to_csv(variance_decomposition_table, variance_decomposition_path, index=False))
written_paths.append(safe_to_csv(variance_decomposition_checks, variance_decomposition_checks_path, index=False))
written_paths.append(safe_to_csv(baseline_benchmarks, baseline_values_path, index=False))
written_paths.append(safe_to_csv(robustness_table, robust_path, index=False))
written_paths.append(safe_to_csv(distributed_summary, distributed_path, index=False))
written_paths.append(safe_to_csv(orthogonal_comparison.reset_index(names="month"), orthogonal_path, index=False))
written_paths.append(safe_save_figure(main_fig, chart_path, dpi=180, bbox_inches="tight"))
written_paths.append(safe_save_figure(change_fig, change_chart_path, dpi=180, bbox_inches="tight"))

model_settings = {
    "notebook_version": "v2",
    "estimation_mode": ESTIMATION_MODE,
    "expectations_source": EXPECTATIONS_SOURCE,
    "expectations_fred_series": FRED_SERIES["exp_1yr"],
    "demand_supply_source": DEMAND_SUPPLY_SOURCE,
    "baseline_mode": BASELINE_MODE,
    "baseline_values": {k: (float(v) if isinstance(v, (int, float, np.floating)) else v) for k, v in baseline_model["baselines"].items()},
    "baseline_model_settings": baseline_model["settings"],
    "coefficient_sample_windows": COEFFICIENT_SAMPLE_WINDOWS,
    "coefficient_sample_metrics": json.loads(coefficient_sample_metrics.to_json(orient="records")),
    "variance_decomposition_method": "share_j = 100 * beta_j * Cov(x_j, pi_12m) / Var(pi_12m); residual = 100 * (1 - R2). Shares can fall outside 0-100 because regressors are not orthogonal.",
    "variance_decomposition_checks": json.loads(variance_decomposition_checks.to_json(orient="records")),
    "change_12m_model_settings": change_baseline_model["settings"],
    "change_12m_sample_start": str(change_level_contributions.index.min()),
    "change_12m_sample_end": str(change_level_contributions.index.max()),
    "change_12m_nobs": int(change_baseline_result["nobs"]),
    "change_12m_r2": float(change_baseline_result["r2"]),
    "change_12m_rmse": float(change_baseline_result["rmse"]),
    "change_12m_sample_metrics": json.loads(change_sample_metrics.to_json(orient="records")),
    "change_12m_variance_decomposition_method": "share_j = 100 * beta_j * Cov(x_j, delta_pi_12m) / Var(delta_pi_12m); residual = 100 * (1 - R2).",
    "change_12m_variance_decomposition_checks": json.loads(change_variance_decomposition_checks.to_json(orient="records")),
    "sample_start": str(baseline_level_contributions.index.min()),
    "sample_end": str(baseline_level_contributions.index.max()),
    "nobs": int(baseline_result["nobs"]),
    "r2": float(baseline_result["r2"]),
    "rmse": float(baseline_result["rmse"]),
    "max_actual_level_identity_error": float(baseline_level_contributions["actual_identity_check"].abs().max()),
    "max_fitted_level_identity_error": float(baseline_level_contributions["fitted_identity_check"].abs().max()),
}
written_paths.append(safe_write_text(json.dumps(model_settings, indent=2), settings_path))

print("Saved:")
for path in written_paths:
    print(path)

## Interpretation Checklist

- This is the v2 notebook that uses `Demand_Supply_Data.xlsx` for demand and supply.
- The main contribution panel is a full inflation-rate decomposition, not a deviation from the 2% target.
- The model constant is a fixed baseline because it is the regression intercept after centering the regressors.
- For each month, `baseline + expectations + demand + supply + lagged_inflation + residual = actual`.
- Positive moving components stack upward from the baseline; negative moving components stack downward from the baseline.
- The fitted line equals `baseline + expectations + demand + supply + lagged_inflation`; adding the residual recovers actual inflation.
- Demand is the contribution from the workbook `DEMAND` series.
- Supply is the contribution from the workbook `SUPPLY` series.
- Expectations are the Cleveland Fed one-year-ahead expected inflation rate from FRED `EXPINF1YR`.
- The main chart uses full-sample coefficients; the coefficient table also reports start-to-January-2020 and January-2020-to-latest estimates.
- The variance decomposition follows the Section 5 covariance formula and assigns the residual share as `100 * (1 - R²)`.
- The 12-month-change section decomposes `pi_12m - pi_12m.shift(12)`, which is closer to the dependent-variable setup used in the reference paper.
- HAC standard errors support inference but do not change fitted contributions.
- Large or persistent residuals suggest omitted factors, measurement differences, insufficient lag structure, or model instability.
- Treat the chart as regression-implied historical accounting, not as structural causal shares.